In [1]:
import pandas as pd
import numpy as np
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException 
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
import re
import time

In [2]:
def set_up_driver():
    chrome_options = Options()
    chrome_options.add_argument("--headless") 
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("start-maximized")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled") # Tắt tính năng phát hiện tự động của trình duyệt
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"]) # Loại bỏ thông báo "Chrome is being controlled by automated test software"
    chrome_options.add_experimental_option('useAutomationExtension', False) # Vô hiệu hóa tiện ích mở rộng tự động hóa
    driver = webdriver.Chrome(options=chrome_options)
    return driver

In [3]:
def get_text(driver, selector):
    try:
        el = driver.find_element(By.CSS_SELECTOR, selector)
        return el.text.strip()
    except:
        return None

In [4]:
def get_section_detail(driver, title):
    try:
        
        # Tìm container chứa nội dung
        container = driver.find_element(By.CSS_SELECTOR, "div.render-html.text-14")
        
        # Lấy tất cả h3 trong container
        h3s = container.find_elements(By.TAG_NAME, "h3")
        title_norm = title.lower().strip()
        
        for h3 in h3s:
            h3_text = h3.get_attribute("textContent") or ""
            
            if title_norm in h3_text.lower():
                # Lấy siblings trực tiếp từ h3
                siblings = h3.find_elements(By.XPATH, "./following-sibling::*")
                
                content = []
                for sib in siblings:
                    tag = sib.tag_name.lower()
                    
                    # Dừng khi gặp h3 tiếp theo
                    if tag == "h3":
                        break
                    
                    # Lấy text
                    t = sib.get_attribute("textContent") or ""
                    t = t.strip()
                    if t:
                        content.append(t)
                
                return "\n".join(content) if content else None
                
    except:
        pass
    return None

In [5]:
def get_product_images(driver):
    images = []

    try:
        img_elements = driver.find_elements(By.CSS_SELECTOR, "img[src*='Products/Images'], img[src*='cdn.tgdd.vn']")

        for img in img_elements:
            src = img.get_attribute("src")
            if src and "Products/Images"in src:
                if "https://cdn.tgdd.vn" in src:
                    match = re.search(r'(https://cdn\.tgdd\.vn/Products/Images/[^\s]+)', src)
                    if match:
                        original_url = match.group(1)
                        if original_url not in images:
                            images.append(original_url)
                elif src not in images:
                    images.append(src)
    except Exception as e:
        print(f"Error {e}")
        pass
    return images

In [6]:
# Lấy thông tin từ bảng grid
def get_product_infomation(driver, label):
    try:
        rows = driver.find_elements(By.CSS_SELECTOR, "#contentRef .grid.grid-cols-3")
        for row in rows:
            label_el = row.find_element(By.CSS_SELECTOR, "div.text-13.font-bold")
            label_text = label_el.get_attribute("textContent") or ""
            
            if label.lower() in label_text.lower():
                value_el = row.find_element(By.CSS_SELECTOR, "div.col-span-2 div.mr-2")
                value_text = value_el.get_attribute("textContent") or ""
                return value_text.strip()
    except Exception as e:
        print(f"Get product information error: {e}")
        pass
    return None

In [7]:
def click_expand_button(driver):
    try:
        xpath = "//*[@id='contentRef']/div/div[3]/button"
        button = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.XPATH, xpath))
        )
        button.click()
        print("Clicked button at div[3]")
        time.sleep(2)
        return True
    except:
        print("div[3] failed, trying other indices...")
    
    for i in range(1, 7):
        if i == 3:  # Bỏ qua div[3] vì đã thử rồi
            continue
        try:
            xpath = f"//*[@id='contentRef']/div/div[{i}]/button"
            button = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.XPATH, xpath))
            )
            button.click()
            print(f"Clicked button at div[{i}]")
            time.sleep(2)
            return True
        except:
            continue
    
    print("No clickable button found")
    return False


In [8]:
def get_medicine_data(driver):
    try:
        WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, "div[class*='text-20'][class*='font-bold']")))
    except:
        pass

    name = get_text(driver, "div.text-20.font-bold")
    if not name:
        try:
            name = driver.title.split(' - ')[0].strip()
        except:
            name = None

    price = None
    try:
        price_el = driver.find_element(By.CSS_SELECTOR, "body > div.bg-secondary-300 > div.w-1200px.mx-auto.min-h-screen.relative.z-1 > div > div.ml-4.flex.h-auto.w-\[540px\].cursor-default.flex-col.gap-20px > div:nth-child(1) > div:nth-child(4) > div > p")
        price = price_el.text.strip()
    except:
        pass
    

    images = get_product_images(driver)


    expand_button_1 = driver.find_element(By.XPATH, "//*[@id='contentRef']/div/div[6]/button")
    expand_button_1.click()
    time.sleep(2)

    cong_dung_tom_tat = get_product_infomation(driver, "Công dụng")
    thanh_phan_chinh = get_product_infomation(driver, "Thành phần chính")
    doi_tuong = get_product_infomation(driver, "Đối tượng sử dụng")
    thuong_hieu = get_product_infomation(driver, "Thương hiệu")
    noi_san_xuat = get_product_infomation(driver, "Nơi sản xuất")
    dang_bao_che = get_product_infomation(driver, "Dạng bào chế")
    han_dung = get_product_infomation(driver, "Hạn dùng")


    close_btn = driver.find_element(By.CSS_SELECTOR, "button[class*='absolute'][class*='right']")
    close_btn.click()
    time.sleep(2)



    # expand_button_2 = driver.find_element(By.XPATH, "//*[@id='contentRef']/div/div[3]/button")
    # expand_button_2.click()

    click_expand_button(driver)
    time.sleep(2)

    thanh_phan_chi_tiet = get_section_detail(driver, "1. Thành phần")
    cong_dung_chi_tiet = get_section_detail(driver, "2. Công dụng")
    cach_dung = get_section_detail(driver, "3. Cách dùng")
    chong_chi_dinh = get_section_detail(driver, "4. Chống chỉ định")
    tac_dung_phu = get_section_detail(driver, "5. Tác dụng phụ")
    luu_y = get_section_detail(driver, "6. Lưu ý")

    return {
        "ten_thuoc": name,
        "gia": price,
        "hinh_anh": "|".join(images) if images else None,
        "cong_dung_tom_tat": cong_dung_tom_tat,
        "thanh_phan_chinh": thanh_phan_chinh,
        "doi_tuong": doi_tuong,
        "thuong_hieu": thuong_hieu,
        "noi_san_xuat": noi_san_xuat,
        "dang_bao_che": dang_bao_che,
        "han_dung": han_dung,
        "thanh_phan_chi_tiet": thanh_phan_chi_tiet,
        "cong_dung_chi_tiet": cong_dung_chi_tiet,
        "cach_dung": cach_dung,
        "chong_chi_dinh": chong_chi_dinh,
        "tac_dung_phu": tac_dung_phu,
        "luu_y": luu_y
    }


<>:16: SyntaxWarning: invalid escape sequence '\['
<>:16: SyntaxWarning: invalid escape sequence '\['
C:\Users\Admin\AppData\Local\Temp\ipykernel_29064\2271683223.py:16: SyntaxWarning: invalid escape sequence '\['
  price_el = driver.find_element(By.CSS_SELECTOR, "body > div.bg-secondary-300 > div.w-1200px.mx-auto.min-h-screen.relative.z-1 > div > div.ml-4.flex.h-auto.w-\[540px\].cursor-default.flex-col.gap-20px > div:nth-child(1) > div:nth-child(4) > div > p")


In [9]:
def crawl_error_links(error_csv, original_csv, out_csv):
    df_error = pd.read_csv(error_csv)
    df_original = pd.read_csv(original_csv)
    
    # Merge để lấy category và drug_type từ file gốc
    df = df_error.merge(
        df_original[['link', 'category', 'drug_type']], 
        left_on='url', 
        right_on='link',
        how='left'
    )

    df = df.drop(columns=['link'])
    
    # Xử lý NaN
    # df['category'] = df['category'].fillna('')
    # df['drug_type'] = df['drug_type'].fillna('')
    driver = set_up_driver()
    results = []

    try:

        for i, row in df.iterrows():
            url = row['url']
            category = row.get('category', '')
            drug_type = row.get('drug_type', '')

            try:
                driver.get(url)
                data = get_medicine_data(driver)
                data['url'] = url
                data['category'] = category
                data['drug_type'] = drug_type
                results.append(data)
                print(f"[{i+1}/{len(df)}] crawled: {data['ten_thuoc']}")
            except Exception as e:
                results.append({'url': url, 'error': str(e)})
                print(f"[{i+1}/{len(df)}] failed: {url} with error {e}")

            time.sleep(2)  

            if (i + 1) % 50 == 0:
                pd.DataFrame(results).to_csv(out_csv, index=False, encoding='utf-8-sig')
                print(f"Saved {len(results)} records to {out_csv}")

        
    finally:
        driver.quit()
        pd.DataFrame(results).to_csv(out_csv, index=False, encoding='utf-8-sig')
        print(f"Đã lưu {len(results)} sản phẩm vào {out_csv}")

 

In [10]:
driver = set_up_driver()
driver.get("https://www.nhathuocankhang.com/thuoc-bo-va-vitamin/cezinco-110mg-5ml-h-30-ong?sku=1193668000849")
time.sleep(3)
data = get_medicine_data(driver)
print(data)
driver.quit()

Clicked button at div[3]
{'ten_thuoc': 'Dung dịch uống Cezinco 110mg/5ml Allomed phòng ngừa và điều trị thiếu vitamin C, thiếu kẽm hộp 30 ống', 'gia': '11.300₫', 'hinh_anh': None, 'cong_dung_tom_tat': 'Phòng ngừa và điều trị thiếu vitamin C, thiếu kẽm', 'thanh_phan_chinh': 'Kẽm, Acid ascorbic', 'doi_tuong': None, 'thuong_hieu': 'Dược phẩm Allomed', 'noi_san_xuat': 'Việt Nam', 'dang_bao_che': 'Dung dịch uống', 'han_dung': '24 tháng kể từ ngày sản xuất', 'thanh_phan_chi_tiet': 'Công thức cho 5ml dung dịch\nAcid ascorbic (dưới dạng natri ascorbat) 100 mg.Kẽm sulfat monohydrat (tương đương 10mg nguyên tố kẽm) 27,44 mg.\nTá dược vừa đủ 5ml (glycerin, propylen glycol, acid citric, methyl paraben, propyl paraben, sucralose, sunset yellow dye, tartrazin yellow dye, natri metabisulfit, hương cam, nước tinh khiết).', 'cong_dung_chi_tiet': 'Phòng ngừa và điều trị thiếu vitamin C và/hoặc thiếu kẽm.', 'cach_dung': '- Cách dùng\nDùng đường uống cùng với thức ăn.\n- Liều dùng\nTrẻ 1-3 tuổi: 2,5 ml/ng

In [11]:
crawl_error_links("ankhang_medicines_error_rows.csv","ankhang_medicine_links.csv", "ankhang_medicines_fix.csv")

Clicked button at div[3]
[1/66] crawled: Dung dịch uống Cezinco 110mg/5ml Allomed phòng ngừa và điều trị thiếu vitamin C, thiếu kẽm hộp 30 ống
Clicked button at div[3]
[2/66] crawled: Dung dịch uống Cezinco 110mg/5ml Allomed phòng ngừa và điều trị thiếu vitamin C, thiếu kẽm hộp 30 ống
Clicked button at div[3]
[3/66] crawled: Magne - B6 Stella Tablet điều trị hạ magnesium (5 vỉ x 10 viên)
Clicked button at div[3]
[4/66] crawled: Magne - B6 Stella Tablet điều trị hạ magnesium (5 vỉ x 10 viên)
div[3] failed, trying other indices...
Clicked button at div[6]
[5/66] crawled: Dung dịch tiêm Zoladex 10,8mg kiểm soát ung thư vú, tuyến tiền liệt hộp 1 bơm tiêm
div[3] failed, trying other indices...
Clicked button at div[6]
[6/66] crawled: Dung dịch tiêm truyền Vitaplex Injection trị thiếu vitamin nhóm B chai 500ml
div[3] failed, trying other indices...
Clicked button at div[6]
[7/66] crawled: Dung dịch tiêm truyền Vitaplex Injection trị thiếu vitamin nhóm B chai 500ml
Clicked button at div[3]
[8

In [14]:
ankhang_medicince_fix = pd.read_csv("ankhang_medicines_fix.csv")
print(ankhang_medicince_fix.shape)

(66, 19)


In [15]:
ankhang_medicince_fix.head(10)

,ten_thuoc,gia,hinh_anh,cong_dung_tom_tat,thanh_phan_chinh,doi_tuong,thuong_hieu,noi_san_xuat,dang_bao_che,han_dung,thanh_phan_chi_tiet,cong_dung_chi_tiet,cach_dung,chong_chi_dinh,tac_dung_phu,luu_y,url,category,drug_type
0,Dung dịch uống Cezinco 110mg/5ml Allomed phòng...,11.300₫,NaN,"Phòng ngừa và điều trị thiếu vitamin C, thiếu kẽm","Kẽm, Acid ascorbic",NaN,Dược phẩm Allomed,Việt Nam,Dung dịch uống,24 tháng kể từ ngày sản xuất,Công thức cho 5ml dung dịch\nAcid ascorbic (dư...,Phòng ngừa và điều trị thiếu vitamin C và/hoặc...,- Cách dùng\nDùng đường uống cùng với thức ăn....,Bệnh nhân quá mẫn với bất cứ thành phần nào củ...,"Vitamin C\nTăng oxalat niệu, buồn nôn hoặc nôn...",NaN,https://www.nhathuocankhang.com/thuoc-bo-va-vi...,NaN,NaN
1,Dung dịch uống Cezinco 110mg/5ml Allomed phòng...,11.300₫,NaN,"Phòng ngừa và điều trị thiếu vitamin C, thiếu kẽm","Kẽm, Acid ascorbic",NaN,Dược phẩm Allomed,Việt Nam,Dung dịch uống,24 tháng kể từ ngày sản xuất,Công thức cho 5ml dung dịch\nAcid ascorbic (dư...,Phòng ngừa và điều trị thiếu vitamin C và/hoặc...,- Cách dùng\nDùng đường uống cùng với thức ăn....,Bệnh nhân quá mẫn với bất cứ thành phần nào củ...,"Vitamin C\nTăng oxalat niệu, buồn nôn hoặc nôn...",NaN,https://www.nhathuocankhang.com/thuoc-bo-va-vi...,NaN,NaN
2,Magne - B6 Stella Tablet điều trị hạ magnesium...,90.000₫,"https://img.tgdd.vn/imgt/ankhang/f_webp,fit_ou...","Điều trị hạ magnesium huyết nặng, các rối loạn...",NaN,Người lớn và trẻ em,Stella,Việt Nam,Viên nén bao phim tan trong ruột,36 tháng kể từ ngày sản xuất,Thành phần hoạt chất:\n\nVitamin B6 (Pyridoxin...,"Điều trị hạ magnesium huyết nặng, riêng biệt h...",- Cách dùng\nMagne - B6 STELLA Tablet được uốn...,Mẫn cảm với bất kỳ thành phần nào của thuốc.\n...,Tăng magnesium huyết ít gặp sau khi uống các m...,NaN,https://www.nhathuocankhang.com/thuoc-bo-va-vi...,NaN,NaN
3,Magne - B6 Stella Tablet điều trị hạ magnesium...,90.000₫,"https://img.tgdd.vn/imgt/ankhang/f_webp,fit_ou...","Điều trị hạ magnesium huyết nặng, các rối loạn...",NaN,Người lớn và trẻ em,Stella,Việt Nam,Viên nén bao phim tan trong ruột,36 tháng kể từ ngày sản xuất,Thành phần hoạt chất:\n\nVitamin B6 (Pyridoxin...,"Điều trị hạ magnesium huyết nặng, riêng biệt h...",- Cách dùng\nMagne - B6 STELLA Tablet được uốn...,Mẫn cảm với bất kỳ thành phần nào của thuốc.\n...,Tăng magnesium huyết ít gặp sau khi uống các m...,NaN,https://www.nhathuocankhang.com/thuoc-bo-va-vi...,NaN,NaN
4,"Dung dịch tiêm Zoladex 10,8mg kiểm soát ung th...",NaN,https://cdn.tgdd.vn/Products/Images/11358/3288...,Dùng để kiểm soát kiểm soát ung thư tiền liệt ...,NaN,Thuốc kê đơn - Sử dụng theo chỉ định của Bác sĩ,AstraZeneca,Anh,Dung dịch tiêm,24 tháng kể từ ngày sản xuất,NaN,NaN,NaN,NaN,NaN,NaN,https://www.nhathuocankhang.com/thuoc-tri-ung-...,NaN,NaN
5,Dung dịch tiêm truyền Vitaplex Injection trị t...,NaN,https://cdn.tgdd.vn/Products/Images/10053/3288...,Điều trị các bệnh nhân bị thiếu hoặc tăng nhu ...,"Dexpanthenol, Vitamin PP, Vitamin C, Vitamin B...",NaN,Siu Guan Chem,Đài Loan,Dung dịch tiêm truyền,36 tháng kể từ ngày sản xuất,NaN,NaN,NaN,NaN,NaN,NaN,https://www.nhathuocankhang.com/thuoc-bo-va-vi...,NaN,NaN
6,Dung dịch tiêm truyền Vitaplex Injection trị t...,NaN,https://cdn.tgdd.vn/Products/Images/10053/3288...,Điều trị các bệnh nhân bị thiếu hoặc tăng nhu ...,"Dexpanthenol, Vitamin PP, Vitamin C, Vitamin B...",NaN,Siu Guan Chem,Đài Loan,Dung dịch tiêm truyền,36 tháng kể từ ngày sản xuất,NaN,NaN,NaN,NaN,NaN,NaN,https://www.nhathuocankhang.com/thuoc-bo-va-vi...,NaN,NaN
7,Vitamin A-D HDpharma điều trị tình trạng thiếu...,NaN,NaN,"Dự phòng và điều trị thiếu vitamin A - D, rối ...","Vitamin D3, Vitamin A",Thuốc kê đơn - Sử dụng theo chỉ định của Bác sĩ,HDPHARMA,Việt Nam,Viên nang mềm,24 tháng kể từ ngày sản xuất,Mỗi viên nang mềm chứa\nHoạt chất:\nVitamin A ...,Dự phòng và điều trị các triệu chứng do thiếu ...,- Cách dùng\nUống trong hoặc ngay sau khi ăn.\...,Mẫn cảm với một trong các thành phần của thuốc...,Dùng Vitamin A-D với liều chỉ địn

In [16]:
# Gộp với file chính
data_fix = pd.read_csv("ankhang_medicines_fix.csv")
data_main = pd.read_csv("ankhang_medicines_data_demo.csv")

data_combined = pd.concat([data_main, data_fix], ignore_index=True)
data_combined.to_csv("ankhang_medicines_data_full.csv", index=False, encoding='utf-8-sig')

In [17]:
data_combined.info()
print(data_combined.shape)
data_combined.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2193 entries, 0 to 2192
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   ten_thuoc            2139 non-null   object
 1   gia                  538 non-null    object
 2   hinh_anh             2102 non-null   object
 3   cong_dung_tom_tat    2139 non-null   object
 4   thanh_phan_chinh     2050 non-null   object
 5   doi_tuong            1817 non-null   object
 6   thuong_hieu          2137 non-null   object
 7   noi_san_xuat         2139 non-null   object
 8   dang_bao_che         2138 non-null   object
 9   han_dung             2119 non-null   object
 10  thanh_phan_chi_tiet  2131 non-null   object
 11  cong_dung_chi_tiet   2102 non-null   object
 12  cach_dung            2117 non-null   object
 13  chong_chi_dinh       2099 non-null   object
 14  tac_dung_phu         2042 non-null   object
 15  luu_y                925 non-null    object
 16  url   

,ten_thuoc,gia,hinh_anh,cong_dung_tom_tat,thanh_phan_chinh,doi_tuong,thuong_hieu,noi_san_xuat,dang_bao_che,han_dung,thanh_phan_chi_tiet,cong_dung_chi_tiet,cach_dung,chong_chi_dinh,tac_dung_phu,luu_y,url,category,drug_type,error
0,"Timi Roitin trị viêm dây thần kinh, thoái hóa ...",528.000₫,https://cdn.tgdd.vn/Products/Images/10053/1304...,"Bổ sung vitamin nhóm B, điều trị viêm dây thần...","Chondroitin, Vitamin B5, Fursultiamine, Vitami...",Người lớn,Phil Inter Pharma,Việt Nam,Viên nang mềm,36 tháng kể từ ngày sản xuất,"Hoạt chất: Chondroitin sulfate natri 90mg, Nic...",- Bổ sung các vitamin nhóm B trong các trường ...,Người lớn: Uống 1 viên/ngày.\n- Quá liều\nDùng...,- Quá mẫn với bất cứ thành phần nào của thuốc....,TIMIROITIN thường được dung nạp tốt khi dùng ở...,- Thận trọng khi sử dụng\n- Khi sử dụng nicoti...,https://www.nhathuocankhang.com/thuoc-bo-va-vi...,Thuốc bổ và vitamin,Không kê đơn,NaN
1,B Complex C bổ sung vitamin nhóm B và vitamin ...,800₫,https://cdn.tgdd.vn/Products/Images/10053/1295...,Dự phòng & bổ sung khi thiếu hụt các vitamin n...,"Vitamin PP, Vitamin C, Vitamin B6, Vitamin B2,...",NaN,Vidipha,Việt Nam,Viên nang cứng,24 tháng kể từ ngày sản xuất,"Hoạt chất: Vitamin B1 15mg, Vitamin B2 10mg, V...",B Complex C dự phòng và bổ sung thiếu hụt các ...,Liều lượng: Trung bình: 1 - 2 viên/ngày.\nCách...,Dị ứng với một trong các thành phần của thuốc....,Dùng liều cao nước tiểu sẽ có màu vàng nhạt (d...,- Thận trọng khi sử dụng\nKhi sử dụng nicotina...,https://www.nhathuocankhang.com/thuoc-bo-va-vi...,Thuốc bổ và vitamin,Không kê đơn,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.nhathuocankhang.com/thuoc-bo-va-vi...,NaN,NaN,Message: no such element: Unable to locate ele...
3,Vitamin PP 50 Pharmedic ngăn ngừa thiếu nicoti...,12.000₫,https://cdn.tgdd.vn/Products/Images/10053/1533...,Bổ sung vào khẩu phần ăn để ngăn ngừa thiếu hụ...,Vitamin PP,Người lớn và trẻ em trên 5 tuổi,Pharmedic,Việt Nam,Viên nén,NaN,Nicotinamid 50mg.\nTá dược 1 viên.,Vitamin PP 50 giúp bổ sung vào khẩu phần ăn để...,"Người lớn: mỗi lần 1 - 2 viên, ngày 3 lần.\nTr...","- Dị ứng với nicotinamid.\n- Bệnh gan nặng, lo...",NaN,- Thận trọng khi sử dụng\nNgười có tiền sử loé...,https://www.nhathuocankhang.com/thuoc-bo-va-vi...,Thuốc bổ và vitamin,Không kê đơn,NaN
4,"Agi-Calci bổ sung canxi, trị loãng xương (20 v...",150.000₫,https://cdn.tgdd.vn/Products/Images/10053/2463...,Bổ sung calci khi thiếu hay tăng nhu cầu calci...,"Calci cacbonat, Vitamin D3",NaN,Agimexpharm,Việt Nam,Viên nén bao phim,24 tháng kể từ ngày sản xuất,Mỗi viên nén bao phim chứa:\nHoạt chất: Calci ...,- Agi-Calci bổ sung calci trong các trường hợp...,Uống thuốc buổi sáng hoặc buổi trưa theo liều ...,"- Tăng calci huyết, calci niệu, sỏi calci, suy...",- Dùng thuốc chứa muối calci qua đường uống có...,- Thận trọng khi sử dụng\nThận trọng khi sử dụ...,https://www.nhathuocankhang.com/thuoc-bo-va-vi...,Thuốc bổ và vitamin,Không kê đơn,NaN
5,Vitamin B1 Mekophar 250mg trị các bệnh thiếu v...,7.000₫,https://cdn.tgdd.vn/Products/Images/10053/1535...,"Điều trị các bệnh do thiếu B1, suy nhược cơ th...",NaN,NaN,Mekophar,Việt Nam,Viên nén bao đường,36 tháng kể từ ngày sản xuất,Hoạt chất: Thiamine nitrate (vitamin B1) 250mg...,- Điều trị các bệnh do thiếu vitamin B1 như: b...,- Theo chỉ dẫn của bác sĩ.\n- Người lớn: uống ...,Quá mẫn với một trong các thành phần của thuốc.,"- Tác dụng phụ rất hiếm khi xảy ra như: ngứa, ...",- Thận trọng khi sử dụng\nThận trọng khi sử dụ...,https://www.nhathuocankhang.com/thuoc-bo-va-vi...,Thuốc bổ và vitamin,Không kê đơn,NaN
6,Dung dịch uống Calciumboston Ascorbic trị thiế...,9.100₫,"https://img.tgdd.vn/imgt/ankhang/f_webp,fit_ou...","Điều trị các tình trạng thiếu calci, vitamin C...","Vitamin PP, Vitamin C, Calci glucoheptonat",Người lớn và trẻ em,Boston Pharma,Việt Nam,Dung dịch,36 tháng kể từ ngày sản xuất,Hoạt chất: Mỗi ml dung dịch thuốc chứa:\n- Cal...,Điều trị các tình trạng thiếu vitamin C và Vit...,- Cách dùng\nCalciu